## Hawaii OSM PostGIS Analysis Notebook 🗺️

In this notebook, you will connect to your PostGIS database, read SQL queries from `.sql` files, and run them one by one.

This notebook follows the same style as the earlier setup notebook. Instead of building a setup function, you will use Python to run completed SQL queries, inspect the results, and visualize spatial outputs.

### 🎯 What This Notebook Does
- Connect to a PostGIS database
- Read SQL queries from `.sql` files
- Run SQL queries one by one
- Display results as GeoDataFrames and tables
- Visualize spatial analysis results

### 🧠 Workflow Reminder
- **SQL does the analysis**
- **Python runs the analysis and displays the results**
- **GeoPandas visualizes the spatial results**

### 📍 Notebook Goal
Use this notebook to run the Hawaii analysis queries and visualize the results. Later, you can create your own notebook based on this one to run analysis for a different place.

---

### ⚙️ Step 0: Select the Correct Python Kernel

Before running any cells, make sure the notebook is using the correct Python environment.

**Check the kernel in the top-right corner of the notebook.**

The correct Python environment is **python-gis-postgis-development (.venv)**  
It may appear with a Python version, for example:  
**python-gis-postgis-development (Python 3.11.15)**

If the kernel is **python-gis-postgis-development (Python 3.11.15)**, you can start running cells below.

Steps to select the correct kernel:
1. Click on the kernel (top right corner of the notebook) if it is not **python-gis-postgis-development (Python 3.11.15)** or if it says "Select Kernel"
2. Select **python-gis-postgis-development (Python 3.11.15)**
3. If you do not see the kernel in the list, click on "Select Another Kernel..."  
    a. Click on Python Environments...  
    b. Select **python-gis-postgis-development (Python 3.11.15)**

Once the correct kernel is selected, you can start running cells below.

### 📚 Step 1: Import Required Libraries

We will use the following tools:

- `geopandas`: to read spatial query results into GeoDataFrames and visualize them
- `pandas`: to work with tabular results when needed
- `psycopg2`: to connect to PostgreSQL/PostGIS
- `pathlib`: to work with file paths more cleanly

**💡 Because the SQL queries return a `geom` column, we will use GeoPandas to load and visualize spatial results.**

In [1]:
import pandas as pd
import geopandas as gpd
import psycopg2
from pathlib import Path

print("Libraries imported!")

Libraries imported!


### 🔌 Step 2: Load Spatial Data Workflow

In the previous notebook, you built a function to set up your PostGIS database and load OpenStreetMap data.

Instead of repeating those steps here, we will reuse that function.

This reinforces an important workflow pattern:

- setup is done once using a reusable function  
- analysis notebooks focus only on running queries and interpreting results  

**💡 This ensures your database is ready before running any SQL queries.**

In [ ]:
from src.postgis_setup import setup_postgis_osm

setup_postgis_osm(
    osm_url="https://download.geofabrik.de/north-america/us/hawaii-latest.osm.pbf"
)

print("Database ready")

### 🔌 Step 3: Connect to the PostGIS Database

Before we can run SQL queries, we need to connect to the database.

In this example, we connect to the Hawaii database created in the earlier setup workflow.

**💡 This connection will be used throughout the notebook!**

In [2]:
conn = psycopg2.connect(
    dbname="hawaii",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)

print("✅ Connected to PostGIS database")

✅ Connected to PostGIS database


### 📂 Step 4: Point to the SQL Files

In this workflow, the SQL queries are stored in separate `.sql` files inside the `sql/examples` folder.

This is useful because:
- SQL stays separate from Python
- queries are easier to edit and reuse
- Python runs the analysis, while SQL performs the analysis

We will define file paths to each query so they can be loaded and executed in Python.

**💡 These file paths will be used throughout the notebook.**

In [12]:
sql_folder = Path("../sql/hawaii")

query_1_file = sql_folder / "01_osm_island_areas.sql"
query_2_file = sql_folder / "02_osm_road_network.sql"
query_3_file = sql_folder / "03_osm_waterway_analysis.sql"
query_4_file = sql_folder / "04_osm_water_body_analysis.sql"
query_5_file = sql_folder / "05_osm_multi_table_analysis.sql"

### ▶️ Step 5: Run Query 1 - Island Areas

This query calculates the area of Hawaiian islands in square kilometers.

The SQL query is stored in a separate `.sql` file. In this step, we read the query into Python, send it to the PostGIS database, and load the result as a GeoDataFrame.

In [ ]:
query_1_file = Path("../sql/hawaii/01_osm_island_areas.sql")

# Read SQL query from file
query_1_sql = query_1_file.read_text(encoding="utf-8")

# Execute query and load result as a GeoDataFrame
query_1_results = gpd.read_postgis(query_1_sql, conn, geom_col="geom")

# Display results
query_1_results

/workspaces/osm-postgis/.venv/lib/python3.11/site-packages/geopandas/io/sql.py:185: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,name,area_sq_km,geom
0,Hawaiʻi,10434.890538,"MULTIPOLYGON (((-156.06188 19.72767, -156.0618..."
1,Maui,1884.013022,"MULTIPOLYGON (((-156.69724 20.92712, -156.6971..."
2,Oʻahu,1550.362864,"MULTIPOLYGON (((-158.28058 21.57442, -158.2804..."
3,Kauaʻi,1438.526778,"MULTIPOLYGON (((-159.78796 22.02861, -159.7878..."
4,Molokaʻi,674.682691,"MULTIPOLYGON (((-157.31053 21.10238, -157.3097..."
5,Lānaʻi,365.148210,"MULTIPOLYGON (((-157.06163 20.8908, -157.06053..."
6,Niʻihau,185.912532,"MULTIPOLYGON (((-160.247 21.84116, -160.24694 ..."
7,Kahoʻolawe,115.074602,"MULTIPOLYGON (((-156.70044 20.52421, -156.7002..."
8,Sand Island,4.512648,"MULTIPOLYGON (((-177.3976 28.19586, -177.39758..."
9,Laysan,4.113830,"MULTIPOLYGON (((-171.74294 25.76082, -171.7429..."


### ▶️ Step 6: Run Query 2 - Road Network Analysis

This query calculates total road length for each island using a spatial join.

In [ ]:
query_2_file = Path("../sql/hawaii/02_osm_road_network.sql")

# Read SQL query from file
query_2_sql = query_2_file.read_text(encoding="utf-8")

# Execute query and load result as a GeoDataFrame
query_2_results = gpd.read_postgis(query_2_sql, conn, geom_col="geom")

# Display results
query_2_results

/workspaces/osm-postgis/.venv/lib/python3.11/site-packages/geopandas/io/sql.py:185: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,island_name,total_road_length_km,island_area_sq_km,geom
0,Hawaiʻi,8737.121748,10434.890538,"MULTIPOLYGON (((-156.06188 19.72767, -156.0618..."
1,Oʻahu,8593.669865,1550.362864,"MULTIPOLYGON (((-158.28058 21.57442, -158.2804..."
2,Maui,3575.196611,1884.013022,"MULTIPOLYGON (((-156.69724 20.92712, -156.6971..."
3,Kauaʻi,2514.269925,1438.526778,"MULTIPOLYGON (((-159.78796 22.02861, -159.7878..."
4,Molokaʻi,938.740255,674.682691,"MULTIPOLYGON (((-157.31053 21.10238, -157.3097..."
5,Lānaʻi,578.996251,365.148210,"MULTIPOLYGON (((-157.06163 20.8908, -157.06053..."
6,Niʻihau,165.709528,185.912532,"MULTIPOLYGON (((-160.247 21.84116, -160.24694 ..."
7,Sand Island,43.151564,2.067985,"MULTIPOLYGON (((-157.88969 21.31211, -157.8896..."
8,Ford Island,41.513879,1.801802,"MULTIPOLYGON (((-157.96894 21.36141, -157.9688..."
9,Kahoʻolawe,34.114476,115.074602,"MULTIPOLYGON (((-156.70044 20.52421, -156.7002..."


### ▶️ Step 7: Run Query 3 - Waterway Analysis

This query summarizes waterways by island, including total length, count, and average length.

In [19]:
query_3_file = sql_folder / "03_osm_waterway_analysis.sql"

# Read SQL query from file
query_3_sql = query_3_file.read_text(encoding="utf-8")

# Execute query and load result as a GeoDataFrame
query_3_results = gpd.read_postgis(query_3_sql, conn, geom_col="geom")

# Display results
query_3_results

/workspaces/osm-postgis/.venv/lib/python3.11/site-packages/geopandas/io/sql.py:185: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,island_name,total_waterway_length_km,number_of_waterways,avg_waterway_length_km,geom
0,Kauaʻi,2657.594245,2943,0.903022,"MULTIPOLYGON (((-159.78796 22.02861, -159.7878..."
1,Maui,2041.213749,1131,1.804787,"MULTIPOLYGON (((-156.69724 20.92712, -156.6971..."
2,Molokaʻi,1209.046833,855,1.414090,"MULTIPOLYGON (((-157.31053 21.10238, -157.3097..."
3,Hawaiʻi,1095.152742,558,1.962639,"MULTIPOLYGON (((-156.06188 19.72767, -156.0618..."
4,Oʻahu,794.877318,674,1.179343,"MULTIPOLYGON (((-158.28058 21.57442, -158.2804..."
5,Lānaʻi,740.622079,421,1.759197,"MULTIPOLYGON (((-157.06163 20.8908, -157.06053..."
6,Kahoʻolawe,177.230555,99,1.790208,"MULTIPOLYGON (((-156.70044 20.52421, -156.7002..."
7,Niʻihau,8.123893,3,2.707964,"MULTIPOLYGON (((-160.247 21.84116, -160.24694 ..."
8,Nīhoa,2.620632,18,0.145591,"MULTIPOLYGON (((-161.92883 23.06432, -161.9287..."


### ▶️ Step 8: Run Query 4 - Water Body Analysis

This query analyzes water bodies by island, including total area, count, and largest water body.

In [20]:
query_4_file = sql_folder / "04_osm_water_body_analysis.sql"

# Read SQL query from file
query_4_sql = query_4_file.read_text(encoding="utf-8")

# Execute query and load result as a GeoDataFrame
query_4_results = gpd.read_postgis(query_4_sql, conn, geom_col="geom")

# Display results
query_4_results

/workspaces/osm-postgis/.venv/lib/python3.11/site-packages/geopandas/io/sql.py:185: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,island_name,total_water_body_area_sq_km,number_of_water_bodies,largest_water_body_name,largest_water_body_area_sq_km,geom
0,Oʻahu,40.247456,445,Pearl Harbor,19.370568,"MULTIPOLYGON (((-158.28058 21.57442, -158.2804..."
1,Kauaʻi,35.568082,467,Alaka‘i Swamp,26.147082,"MULTIPOLYGON (((-159.78796 22.02861, -159.7878..."
2,Ford Island,19.370568,1,Pearl Harbor,19.370568,"MULTIPOLYGON (((-157.96894 21.36141, -157.9688..."
3,Niʻihau,6.269016,21,Hālaliʻi Lake,2.693317,"MULTIPOLYGON (((-160.247 21.84116, -160.24694 ..."
4,Molokaʻi,5.516803,78,NaN,3.400520,"MULTIPOLYGON (((-157.31053 21.10238, -157.3097..."
5,Maui,4.016975,255,Keālia Pond,1.072095,"MULTIPOLYGON (((-156.69724 20.92712, -156.6971..."
6,Hawaiʻi,3.465825,392,NaN,0.515854,"MULTIPOLYGON (((-156.06188 19.72767, -156.0618..."
7,Laysan,0.691238,1,NaN,0.691238,"MULTIPOLYGON (((-171.74294 25.76082, -171.7429..."
8,Lānaʻi,0.070397,17,NaN,0.010045,"MULTIPOLYGON (((-157.06163 20.8908, -157.06053..."
9,Southeast Island,0.038638,4,NaN,0.026077,"MULTIPOLYGON (((-175.82152 27.78977, -175.8215..."


### ▶️ Step 9: Run Query 5 - Multi-Table Analysis

This query combines multiple datasets to create an island-level infrastructure summary.

In [23]:
query_5_file = sql_folder / "05_osm_multi_table_analysis.sql"

# Read SQL query from file
query_5_sql = query_5_file.read_text(encoding="utf-8")

# Execute query and load result as a GeoDataFrame
query_5_results = gpd.read_postgis(query_5_sql, conn, geom_col="geom")

# Display results
query_5_results

/workspaces/osm-postgis/.venv/lib/python3.11/site-packages/geopandas/io/sql.py:185: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,island_name,island_area_sq_km,total_road_length_km,total_waterway_length_km,total_water_body_area_sq_km,road_density_km_per_sq_km,geom
0,Sand Island,2.067985,61.761813,0.000000,0.036465,29.865693,"MULTIPOLYGON (((-157.88969 21.31211, -157.8896..."
1,Ford Island,1.801802,41.513879,0.000000,19.370568,23.040197,"MULTIPOLYGON (((-157.96894 21.36141, -157.9688..."
2,Sand Island,4.512648,61.761813,0.000000,0.036465,13.686379,"MULTIPOLYGON (((-177.3976 28.19586, -177.39758..."
3,Oʻahu,1550.362864,8593.669865,794.877318,40.247456,5.543005,"MULTIPOLYGON (((-158.28058 21.57442, -158.2804..."
4,Cutter's Island,0.002601,0.010035,0.000000,0.000000,3.858185,"MULTIPOLYGON (((-158.08815 21.33549, -158.0881..."
5,Eastern Island,1.340968,4.321776,0.000000,0.000000,3.222879,"MULTIPOLYGON (((-177.3471 28.20778, -177.34675..."
6,Maui,1884.013022,3575.196611,2041.213749,4.016975,1.897650,"MULTIPOLYGON (((-156.69724 20.92712, -156.6971..."
7,Kauaʻi,1438.526778,2514.269925,2657.594245,35.568082,1.747809,"MULTIPOLYGON (((-159.78796 22.02861, -159.7878..."
8,Lānaʻi,365.148210,578.996251,740.622079,0.070397,1.585647,"MULTIPOLYGON (((-157.06163 20.8908, -157.06053..."
9,Molokaʻi,674.682691,938.740255,1209.046833,5.516803,1.391380,"MULTIPOLYGON (((-157.31053 21.10238, -157.3097..."


### 🔍 Step 10: Compare and Interpret the Results

Now that all queries have been run, review the outputs and look for patterns.

For example:
- Which island has the largest area?
- Which island has the highest total road length?
- Which island has the greatest road density?
- Which island has the most extensive waterways?
- What patterns do you notice across the different analyses?

This notebook is not only for running queries. It is also a space for inspecting results and preparing for interpretation or visualization.

In [ ]:
conn.close()
print("Database connection closed")